In [14]:
import numpy as np
import pandas as pd
import os
import dotenv
import psycopg
from sqlalchemy import create_engine

import plotly.express as px
import plotly.figure_factory as ff

import dash
from dash import dcc, html, Dash
from dash.dependencies import Input, Output
import dash_bootstrap_components as dbc

In [15]:
# Connect to our deployed Neon DB
dotenv.load_dotenv()
NEON_DATABASE_URL_UNPOOLED = os.getenv('NEON_DATABASE_URL_UNPOOLED')
NEON_DATABASE_URL_UNPOOLED = NEON_DATABASE_URL_UNPOOLED.replace('postgresql://', 'postgresql+psycopg://')
engine = create_engine(NEON_DATABASE_URL_UNPOOLED, pool_pre_ping=True)

In [16]:
b = 'M001239'

In [21]:
# subquery test

myquery = f'''
        SELECT 
            vc.icpsr_comparison_member,
            vc.agree
        FROM vote_compare vc
        WHERE icpsr_member = (
            SELECT icpsr
            FROM crosswalk
            WHERE bioguide_id = '{b}'
        )    
'''
scatter_data = pd.read_sql_query(myquery, con=engine)

DatabaseError: Execution failed on sql '
        SELECT 
            vc.icpsr_comparison_member,
            vc.agree
        FROM vote_compare vc
        WHERE icpsr_member = (
            SELECT icpsr
            FROM crosswalk
            WHERE bioguide_id = 'M001239'
        )    
': (psycopg.errors.CardinalityViolation) more than one row returned by a subquery used as an expression
[SQL: 
        SELECT 
            vc.icpsr_comparison_member,
            vc.agree
        FROM vote_compare vc
        WHERE icpsr_member = (
            SELECT icpsr
            FROM crosswalk
            WHERE bioguide_id = 'M001239'
        )    
]
(Background on this error at: https://sqlalche.me/e/20/f405)

In [22]:
scatter_data

,icpsr_member,icpsr_comparison_member,agree
0,14226,14435,0.028047
1,14226,14858,0.083064
2,14226,14871,0.059331
3,14226,14921,0.800431
4,14226,15021,0.121899
...,...,...,...
215349,99912,42505,1.000000
215350,99912,42506,1.000000
215351,99912,49308,0.000000
215352,99912,49703,0.972414


In [22]:
myquery = f'''
        SELECT *
        FROM vote_compare
        WHERE bioname = (
            SELECT full_name
            FROM members
            WHERE bioguide_id = '{b}'
        )
'''
scatter_data = pd.read_sql_query(myquery, con=engine)

In [23]:
scatter_data

,bioname,comparison_member,agree


In [15]:
myquery = f'''
        SELECT 
            m.full_name,
            vc.agree,
            m.left_right_ideology,
            m.state,
            m.district,
            m.party
        FROM vote_compare vc
        INNER JOIN members m
                    ON vc.comparison_member = m.full_name
        WHERE bioname = (
            SELECT full_name
            FROM members
            WHERE bioguide_id = '{b}'
        )
'''
scatter_data = pd.read_sql_query(myquery, con=engine)

In [14]:
scatter_data

,full_name,agree,left_right_ideology,state,district,party


In [11]:
fig = px.scatter(scatter_data,
                 x = 'left_right_ideology',
                 y = 'agree')

In [12]:
fig

All above can be deleted

In [4]:
os.chdir('..')
from contrans import contrans
ct = contrans()

In [5]:
ct.get_vote_similarity_data()

In [6]:
ct.make_others()

In [7]:
ct.create_new_db()

In [8]:
ct.data_to_postgres()

In [10]:
import subprocess

In [9]:
def deploy_neon_dump():
    with open('contrans.dump', 'wb') as dump:
        result = subprocess.run(['docker', 'compose', 'exec', '-T', 'postgres',
                                 'pg_dump', '-U', 'postgres', '-d', 'contrans', '-Fc', '-v'],
                                stdout=dump, stderr=subprocess.PIPE,
                                encoding='utf-8', errors='replace')
    print(result.stderr)          
    result.check_returncode()

def deploy_neon_restore():
    with open('contrans.dump', 'rb') as dump:                # replaces "< contrans.dump"
        result = subprocess.run(['docker', 'compose', 'exec', '-T', 'postgres', 'sh', '-c',
                                 'pg_restore -v --no-owner --no-privileges --clean --if-exists '
                                 '--single-transaction -d "$NEON_DATABASE_URL_UNPOOLED"'],
                                stdin=dump, capture_output=True,
                                encoding='utf-8', errors='replace')
    print(result.stdout + result.stderr)
    result.check_returncode()

In [11]:
deploy_neon_dump()

pg_dump: executing SELECT pg_catalog.set_config('search_path', '', false);
pg_dump: last built-in OID is 16383
pg_dump: reading extensions
pg_dump: identifying extension members
pg_dump: reading schemas
pg_dump: reading user-defined tables
pg_dump: reading user-defined functions
pg_dump: reading user-defined types
pg_dump: reading procedural languages
pg_dump: reading user-defined aggregate functions
pg_dump: reading user-defined operators
pg_dump: reading user-defined access methods
pg_dump: reading user-defined operator classes
pg_dump: reading user-defined operator families
pg_dump: reading user-defined text search parsers
pg_dump: reading user-defined text search templates
pg_dump: reading user-defined text search dictionaries
pg_dump: reading user-defined text search configurations
pg_dump: reading user-defined foreign-data wrappers
pg_dump: reading user-defined foreign servers
pg_dump: reading default privileges
pg_dump: reading user-defined collations
pg_dump: reading user-defin

In [12]:
deploy_neon_restore()

pg_restore: connecting to database for restore
pg_restore: executing SELECT pg_catalog.set_config('search_path', '', false);
pg_restore: dropping TABLE vote_compare
pg_restore: dropping TABLE terms
pg_restore: dropping TABLE members
pg_restore: dropping TABLE expenditures
pg_restore: dropping TABLE crosswalk
pg_restore: dropping TABLE committees
pg_restore: dropping TABLE committee_members
pg_restore: dropping TABLE bills
pg_restore: creating TABLE "public.bills"
pg_restore: creating TABLE "public.committee_members"
pg_restore: creating TABLE "public.committees"
pg_restore: creating TABLE "public.crosswalk"
pg_restore: creating TABLE "public.expenditures"
pg_restore: creating TABLE "public.members"
pg_restore: creating TABLE "public.terms"
pg_restore: creating TABLE "public.vote_compare"
pg_restore: processing data for table "public.bills"
pg_restore: processing data for table "public.committee_members"
pg_restore: processing data for table "public.committees"
pg_restore: processing da

In [ ]:
congress ='119'
url = f'https://voteview.com/static/data/out/votes/HS{congress}_votes.csv'
votes = pd.read_csv(url)
votes = votes.drop(['congress', 'prob'], axis=1)
vote_compare = pd.merge(votes, votes,
                on = ['chamber', 'rollnumber'],
                how = 'inner')

In [32]:
vote_compare

,chamber,rollnumber,icpsr_x,cast_code_x,icpsr_y,cast_code_y
0,House,1,14854,1,14854,1
1,House,1,14854,1,14863,1
2,House,1,14854,1,14873,6
3,House,1,14854,1,15029,6
4,House,1,14854,1,15433,6
...,...,...,...,...,...,...
137531643,Senate,927,49703,1,42506,1
137531644,Senate,927,49703,1,42507,1
137531645,Senate,927,49703,1,42508,1
137531646,Senate,927,49703,1,49308,6


In [33]:
vote_compare = vote_compare.query("icpsr_x != icpsr_y")
vote_compare['agree'] = vote_compare['cast_code_x'] == vote_compare['cast_code_y']
vote_compare = vote_compare.groupby(['icpsr_x', 'icpsr_y']).agg({'agree': 'mean'}).reset_index()
vote_compare


,icpsr_x,icpsr_y,agree
0,14226,14435,0.028047
1,14226,14858,0.083064
2,14226,14871,0.059331
3,14226,14921,0.800431
4,14226,15021,0.121899
...,...,...,...
215349,99912,42505,1.000000
215350,99912,42506,1.000000
215351,99912,49308,0.000000
215352,99912,49703,0.972414


In [ ]:

vote_compare = vote_compare.rename({'icpsr_x': 'icpsr_member',
                                'icpsr_y': 'icpsr_comparison_member'}, axis=1)
vote_compare.to_parquet('data/raw/vote_compare.parquet', index=False)